# Crystal VAE — Test Evaluation on Carbon-24

**Dataset**: `carbon_24/` (10k carbon structures, 6-24 atoms per unit cell)

**Pipeline**:
1. Load `carbon_24/train.csv` → train VAE
2. Load `carbon_24/test.csv`  → evaluate reconstruction quality
3. Report metrics (MSE, MAE, R²) on held-out test set
4. Generate new crystal parameters from latent space
5. Save a generated crystal as CIF

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pymatgen.core import Structure, Lattice

torch.manual_seed(42)
np.random.seed(42)
print('All libraries loaded.')

All libraries loaded.


## 2. Feature Extraction Helper

In [2]:
def extract_features(df):
    """Parse CIF strings and return (X, y) arrays."""
    features, targets = [], []
    cif_col = [c for c in df.columns if 'cif' in c.lower()][0]
    for i in range(len(df)):
        try:
            struct = Structure.from_str(df[cif_col].iloc[i], fmt='cif')
            a, b, c = struct.lattice.abc
            al, be, ga = struct.lattice.angles
            features.append([len(struct), struct.volume, a, b, c, al, be, ga])
            targets.append(df['energy_per_atom'].iloc[i])
        except:
            continue
    return np.array(features), np.array(targets)

print('Helper defined.')

Helper defined.


## 3. Load & Prepare Training Data

In [3]:
print('Loading carbon_24/train.csv ...')
train_df = pd.read_csv('carbon_24/train.csv')
print(f'Train rows: {len(train_df)}')

X_train, y_train = extract_features(train_df)
print(f'Train features extracted: {X_train.shape}')

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_train_t  = torch.tensor(X_train_sc, dtype=torch.float32)
print('Data normalised and converted to tensor.')

Loading carbon_24/train.csv ...
Train rows: 6091


C:\Users\Dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pymatgen\core\structure.py:3109: UserWarning: Issues encountered while parsing CIF: 1 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


C:\Users\Dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pymatgen\core\structure.py:3109: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


Train features extracted: (6091, 8)
Data normalised and converted to tensor.


## 4. Load Test Data

In [4]:
print('Loading carbon_24/test.csv ...')
test_df = pd.read_csv('carbon_24/test.csv')
print(f'Test rows: {len(test_df)}')

X_test, y_test = extract_features(test_df)
print(f'Test features extracted: {X_test.shape}')

X_test_sc = scaler.transform(X_test)   # use train scaler!
X_test_t  = torch.tensor(X_test_sc, dtype=torch.float32)
print('Test data normalised.')

Loading carbon_24/test.csv ...
Test rows: 2030


C:\Users\Dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pymatgen\core\structure.py:3109: UserWarning: Issues encountered while parsing CIF: 1 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


Test features extracted: (2030, 8)
Test data normalised.


## 5. VAE Model Definition

In [5]:
class CrystalVAE(nn.Module):
    def __init__(self, input_dim=8, latent_dim=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 32),        nn.ReLU()
        )
        self.mu     = nn.Linear(32, latent_dim)
        self.logvar = nn.Linear(32, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32), nn.ReLU(),
            nn.Linear(32, 64),         nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.mu(h), self.logvar(h)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar


def vae_loss(recon, x, mu, logvar, beta=1.0):
    recon_loss = nn.MSELoss()(recon, x)
    kl_loss    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl_loss, recon_loss, kl_loss


model = CrystalVAE(input_dim=8, latent_dim=4)
print(model)

CrystalVAE(
  (encoder): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
  )
  (mu): Linear(in_features=32, out_features=4, bias=True)
  (logvar): Linear(in_features=32, out_features=4, bias=True)
  (decoder): Sequential(
    (0): Linear(in_features=4, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=8, bias=True)
  )
)


## 6. Train VAE on carbon_24 Train Set

In [6]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 300

train_losses = []
for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    recon, mu, logvar = model(X_train_t)
    loss, r_loss, k_loss = vae_loss(recon, X_train_t, mu, logvar)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())
    if epoch % 50 == 0:
        print(f'Epoch {epoch:>3} | Total={loss.item():.4f}  Recon={r_loss.item():.4f}  KL={k_loss.item():.4f}')

print('\nTraining complete.')

Epoch   0 | Total=1.0355  Recon=1.0215  KL=0.0140


Epoch  50 | Total=0.9903  Recon=0.9768  KL=0.0135


Epoch 100 | Total=0.9081  Recon=0.8016  KL=0.1065


Epoch 150 | Total=0.8889  Recon=0.7721  KL=0.1168


Epoch 200 | Total=0.8679  Recon=0.6943  KL=0.1736


Epoch 250 | Total=0.8719  Recon=0.6890  KL=0.1829



Training complete.


## 7. Evaluate Reconstruction on Test Set

In [7]:
model.eval()
with torch.no_grad():
    recon_test, mu_test, logvar_test = model(X_test_t)
    test_loss, test_r, test_kl = vae_loss(recon_test, X_test_t, mu_test, logvar_test)

# Back to original scale for interpretable metrics
recon_orig = scaler.inverse_transform(recon_test.numpy())

mse = mean_squared_error(X_test, recon_orig)
mae = mean_absolute_error(X_test, recon_orig)
r2  = r2_score(X_test, recon_orig)

print('='*45)
print(f'  Test Total Loss  : {test_loss.item():.4f}')
print(f'  Test Recon Loss  : {test_r.item():.4f}')
print(f'  Test KL Loss     : {test_kl.item():.4f}')
print('-'*45)
print(f'  MSE  (raw scale) : {mse:.4f}')
print(f'  MAE  (raw scale) : {mae:.4f}')
print(f'  R²   (raw scale) : {r2:.4f}')
print('='*45)

# Per-feature MAE
feat_names = ['num_atoms','volume','a','b','c','alpha','beta','gamma']
per_feat_mae = np.abs(X_test - recon_orig).mean(axis=0)
print('\nPer-feature MAE:')
for name, val in zip(feat_names, per_feat_mae):
    print(f'  {name:>10}: {val:.4f}')

  Test Total Loss  : 0.8599
  Test Recon Loss  : 0.6679
  Test KL Loss     : 0.1920
---------------------------------------------
  MSE  (raw scale) : 115.5101
  MAE  (raw scale) : 6.5055
  R²   (raw scale) : 0.3338

Per-feature MAE:
   num_atoms: 1.6520
      volume: 10.3473
           a: 0.4164
           b: 0.6953
           c: 1.0356
       alpha: 14.2436
        beta: 11.2538
       gamma: 12.4001


## 8. Generate New Crystal Structures from Latent Space

In [8]:
N_GEN = 5
with torch.no_grad():
    z = torch.randn(N_GEN, 4)
    gen_sc = model.decoder(z).numpy()

generated = scaler.inverse_transform(gen_sc)

print(f'Generated {N_GEN} crystal parameter sets:\n')
print(f'{"#":>3}  {"atoms":>6}  {"volume":>8}  {"a":>7}  {"b":>7}  {"c":>7}  {"alpha":>7}  {"beta":>7}  {"gamma":>7}')
for i, g in enumerate(generated):
    print(f'{i:>3}  {g[0]:>6.1f}  {g[1]:>8.3f}  {g[2]:>7.4f}  {g[3]:>7.4f}  {g[4]:>7.4f}  {g[5]:>7.3f}  {g[6]:>7.3f}  {g[7]:>7.3f}')

Generated 5 crystal parameter sets:

  #   atoms    volume        a        b        c    alpha     beta    gamma
  0    14.8    93.109   2.6835   5.3495   7.5803   93.375   92.383   91.755
  1     6.8    42.898   2.7446   3.7093   5.0525   79.860   81.641   79.069
  2     7.2    45.180   2.8551   3.8948   5.1486  100.034   98.737  100.031
  3    12.2    78.436   2.8803   4.9419   6.8021  105.727  100.017  100.731
  4     7.8    50.040   2.9987   4.1745   5.3074   72.517   75.730   74.195


## 9. Save Best Generated Crystal as CIF

In [9]:
best = generated[0]
num_atoms, volume, a, b, c, alpha, beta, gamma = best

# clamp angles to valid range
alpha = np.clip(alpha, 60, 150)
beta  = np.clip(beta,  60, 150)
gamma = np.clip(gamma, 60, 150)
a, b, c = max(a, 1.0), max(b, 1.0), max(c, 1.0)

lattice = Lattice.from_parameters(a, b, c, alpha, beta, gamma)
n       = int(max(1, round(num_atoms)))
coords  = np.random.rand(n, 3)
species = ['C'] * n

structure = Structure(lattice, species, coords)
out_path  = 'generated_carbon24.cif'
structure.to(filename=out_path)
print(f'Crystal saved → {out_path}')
print(f'  Lattice: a={a:.4f} b={b:.4f} c={c:.4f}')
print(f'  Angles : α={alpha:.2f}° β={beta:.2f}° γ={gamma:.2f}°')
print(f'  Atoms  : {n} Carbon atoms')

Crystal saved → generated_carbon24.cif
  Lattice: a=2.6835 b=5.3495 c=7.5803
  Angles : α=93.37° β=92.38° γ=91.76°
  Atoms  : 15 Carbon atoms


## ✅ Test Complete

The VAE was trained on `carbon_24/train.csv` and evaluated on `carbon_24/test.csv`.
Metrics (MSE, MAE, R²) are reported above along with per-feature reconstruction accuracy.